<a href="https://colab.research.google.com/github/LouiseHjorth/speciale-forecasting-hay/blob/main/Hay_forecast_bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Indlæsning af data

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np

file_name = list(uploaded.keys())[0]
print("Fil valgt:", file_name)

df = pd.read_excel(file_name, sheet_name="Ark1")

# Omdømmer kolonner

In [ ]:
rename = {
    "Date ": "date",
    "Size": "Size",
    "Faktisk solgt": "actual",
    "Hay nuværende ": "hay_current",
    "Baseline": "baseline",
    "Arima ": "arima",
    "Holt winther": "holt_winters",
    "Elastic net": "elastic_net",
    "RF": "random_forest",
    "Xgboost": "xgboost",
}

df = df.rename(columns=rename)

df["date"] = pd.to_datetime(df["date"])

df = df.dropna(subset=["Size", "actual"])

# Modeller

In [ ]:
models = {
    "hay_current": "HAY nuværende",
    "baseline": "Baseline",
    "arima": "ARIMA",
    "holt_winters": "Holt-Winters",
    "elastic_net": "Elastic Net",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost"
}


models = {k: v for k, v in models.items() if k in df.columns}


for model in models.keys():
    df[model] = df[model].clip(lower=0)

# Langt format

In [ ]:
long_df = df.melt(
    id_vars=["date", "Size", "actual"],
    value_vars=list(models.keys()),
    var_name="model",
    value_name="forecast"
)

long_df["model_name"] = long_df["model"].map(models)

# Beregn forecast fejl

In [ ]:
long_df["error"] = long_df["forecast"] - long_df["actual"]

long_df["absolute_error"] = long_df["error"].abs()

long_df["overforecast"] = long_df["error"].clip(lower=0)

long_df["underforecast"] = (-long_df["error"]).clip(lower=0)

# Opsummer pr size og model

In [ ]:
impact = (
    long_df
    .groupby(["Size", "model", "model_name"])
    .agg(
        observations=("actual", "count"),
        total_actual=("actual", "sum"),
        total_forecast=("forecast", "sum"),
        MAE=("absolute_error", "mean"),
        total_overforecast=("overforecast", "sum"),
        total_underforecast=("underforecast", "sum"),
        avg_overforecast=("overforecast", "mean"),
        avg_underforecast=("underforecast", "mean")
    )
    .reset_index()
)

# Sammenligner med Hay nuværende metode

In [ ]:
hay_baseline = (
    impact[impact["model"] == "hay_current"]
    [["Size", "total_overforecast", "total_underforecast", "MAE"]]
    .rename(columns={
        "total_overforecast": "hay_overforecast",
        "total_underforecast": "hay_underforecast",
        "MAE": "hay_MAE"
    })
)

impact = impact.merge(hay_baseline, on="Size", how="left")

impact["overforecast_reduction_units"] = (
    impact["hay_overforecast"] - impact["total_overforecast"]
)

impact["overforecast_reduction_pct"] = (
    impact["overforecast_reduction_units"] / impact["hay_overforecast"] * 100
)

impact["MAE_improvement_units"] = (
    impact["hay_MAE"] - impact["MAE"]
)

impact["MAE_improvement_pct"] = (
    impact["MAE_improvement_units"] / impact["hay_MAE"] * 100
)

numeric_cols = impact.select_dtypes(include="number").columns
impact[numeric_cols] = impact[numeric_cols].round(2)


# Finder bedste model pr Size

In [ ]:
best_models = (
    impact[impact["model"] != "hay_current"]
    .sort_values(["Size", "overforecast_reduction_units"], ascending=[True, False])
    .groupby("Size")
    .head(1)
)


# Printer resultater

In [ ]:
print("\nSAMLET IMPACT-ANALYSE")
display(impact)

print("\nBEDSTE MODEL PR. Size")
display(best_models)

# Gemmer til excel

In [ ]:
output_file = "forecast_inventory_impact_analysis.xlsx"

with pd.ExcelWriter(output_file) as writer:
    impact.to_excel(writer, sheet_name="Alle modeller", index=False)
    best_models.to_excel(writer, sheet_name="Bedste modeller", index=False)
    long_df.to_excel(writer, sheet_name="Beregninger", index=False)

print(f"\nResultater gemt som: {output_file}")

files.download(output_file)